# MonoDETR teacher feasibility benchmark

This notebook evaluates the official MonoDETR checkpoint on the exact MobileADAS3D Chen validation split. It is a transfer-learning gate, not a student-training notebook. A report is valid only when all 3,769 validation prediction files exist and `complete_split` is true.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, shlex, shutil, subprocess, sys, time, yaml

MOBILE_REPO_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
MOBILE_REPO = Path('/content/mobile_adas3d')
MONODETR_REPO = Path('/content/MonoDETR')
MONODETR_COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
MONODETR_KITTI = Path('/content/monodetr_kitti')
TEACHER_ROOT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/teachers/monodetr')
TEACHER_CHECKPOINT = TEACHER_ROOT / 'checkpoint_best.pth'
TEACHER_OUTPUT = TEACHER_ROOT / 'chen_val_20260729'
OFFICIAL_CHECKPOINT_GDRIVE_ID = '1d8fbAt-CQF-IN8UEHuw3NimmfONhH6iA'

def run_streamed(command, cwd=None, env=None):
    command = [str(value) for value in command]
    print('+', shlex.join(command), flush=True)
    merged_env = os.environ.copy()
    if env:
        merged_env.update({str(k): str(v) for k, v in env.items()})
    process = subprocess.Popen(
        command, cwd=cwd, env=merged_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Command failed with exit code {return_code}: {shlex.join(command)}')

run_streamed(['nvidia-smi'])
if not MOBILE_REPO.exists():
    run_streamed(['git', 'clone', MOBILE_REPO_URL, MOBILE_REPO])
else:
    run_streamed(['git', 'pull', '--ff-only'], cwd=MOBILE_REPO)
print('MobileADAS3D commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=MOBILE_REPO, text=True).strip())
print('Mobile repo:', MOBILE_REPO)
print('Drive KITTI root:', DRIVE_DATASET_ROOT)
print('Optional local KITTI root:', LOCAL_DATASET_ROOT)
print('Split dir:', SPLIT_DIR)
print('Teacher output:', TEACHER_OUTPUT)

In [ ]:
# Resolve KITTI from the already staged local copy when available; otherwise
# use Google Drive directly. Drive may use image_02/label_02 aliases.
SOURCE_CANDIDATES = {
    'image_2': ('training/image_2', 'training/image_02'),
    'label_2': ('training/label_2', 'training/label_02'),
    'calib': ('training/calib',),
}

def resolve_kitti_source(root):
    resolved = {}
    missing = []
    for canonical_name, candidates in SOURCE_CANDIDATES.items():
        source = next((root / candidate for candidate in candidates if (root / candidate).is_dir()), None)
        if source is None:
            missing.append(f'{canonical_name}: {list(candidates)}')
        else:
            resolved[canonical_name] = source
    return resolved, missing

local_sources, local_missing = resolve_kitti_source(LOCAL_DATASET_ROOT)
drive_sources, drive_missing = resolve_kitti_source(DRIVE_DATASET_ROOT)
if not local_missing:
    KITTI_SOURCE_ROOT = LOCAL_DATASET_ROOT
    KITTI_SOURCES = local_sources
    print('Using existing local staged KITTI data.')
elif not drive_missing:
    KITTI_SOURCE_ROOT = DRIVE_DATASET_ROOT
    KITTI_SOURCES = drive_sources
    print('Local stage not found; using mounted Google Drive directly.')
else:
    print('Drive root entries:', sorted(path.name for path in DRIVE_DATASET_ROOT.iterdir()) if DRIVE_DATASET_ROOT.is_dir() else 'root missing')
    training_dir = DRIVE_DATASET_ROOT / 'training'
    print('Drive training entries:', sorted(path.name for path in training_dir.iterdir()) if training_dir.is_dir() else 'training missing')
    raise FileNotFoundError(
        f'KITTI folders were not found locally or in Drive. Drive missing: {drive_missing}'
    )

for split_path in (SPLIT_DIR / 'train.txt', SPLIT_DIR / 'val.txt'):
    if not split_path.is_file():
        raise FileNotFoundError(f'Missing Chen split file: {split_path}')
print('Resolved KITTI source root:', KITTI_SOURCE_ROOT)
for name, path in KITTI_SOURCES.items():
    print(f'  {name}: {path}')

(MONODETR_KITTI / 'training').mkdir(parents=True, exist_ok=True)
(MONODETR_KITTI / 'ImageSets').mkdir(parents=True, exist_ok=True)

def ensure_symlink(link, target):
    if link.is_symlink() and link.resolve() == target.resolve():
        return
    if link.exists() or link.is_symlink():
        raise RuntimeError(f'Refusing to replace existing path: {link}')
    link.symlink_to(target, target_is_directory=True)

for name in ('image_2', 'label_2', 'calib'):
    ensure_symlink(MONODETR_KITTI / 'training' / name, KITTI_SOURCES[name])
shutil.copy2(SPLIT_DIR / 'train.txt', MONODETR_KITTI / 'ImageSets/train.txt')
shutil.copy2(SPLIT_DIR / 'val.txt', MONODETR_KITTI / 'ImageSets/val.txt')

train_ids = (MONODETR_KITTI / 'ImageSets/train.txt').read_text().splitlines()
val_ids = (MONODETR_KITTI / 'ImageSets/val.txt').read_text().splitlines()
print('Train IDs:', len(train_ids))
print('Val IDs:', len(val_ids))
if len(train_ids) != 3712 or len(val_ids) != 3769:
    raise RuntimeError('Unexpected Chen split counts; stop before teacher evaluation.')

In [ ]:
# Clone the official implementation at a pinned commit and compile its CUDA op.
if not MONODETR_REPO.exists():
    run_streamed(['git', 'clone', 'https://github.com/ZrrSkywalker/MonoDETR.git', MONODETR_REPO])
run_streamed(['git', 'fetch', '--all'], cwd=MONODETR_REPO)
run_streamed(['git', 'checkout', MONODETR_COMMIT], cwd=MONODETR_REPO)
run_streamed([
    sys.executable, '-m', 'pip', 'install', '-q',
    'gdown', 'pyyaml', 'scipy', 'opencv-python-headless',
    'numba', 'scikit-image', 'tqdm', 'ninja',
])

# The pinned MonoDETR extension predates PyTorch's ScalarType dispatch API.
# Patch only the two deprecated dispatch expressions, and verify the exact
# count so an upstream/source-layout change cannot be silently modified.
ops_dir = MONODETR_REPO / 'lib/models/monodetr/ops'
cuda_source = ops_dir / 'src/cuda/ms_deform_attn_cuda.cu'
deprecated_dispatch = 'AT_DISPATCH_FLOATING_TYPES(value.type(),'
compatible_dispatch = 'AT_DISPATCH_FLOATING_TYPES(value.scalar_type(),'
cuda_text = cuda_source.read_text()
deprecated_count = cuda_text.count(deprecated_dispatch)
compatible_count = cuda_text.count(compatible_dispatch)
if deprecated_count == 2 and compatible_count == 0:
    cuda_source.write_text(cuda_text.replace(deprecated_dispatch, compatible_dispatch))
    print('Applied PyTorch ScalarType compatibility patch to both CUDA dispatch calls.')
elif deprecated_count == 0 and compatible_count == 2:
    print('PyTorch ScalarType compatibility patch is already applied.')
else:
    raise RuntimeError(
        'Unexpected MonoDETR CUDA source: expected exactly two old or two patched '
        f'dispatch calls, found old={deprecated_count}, patched={compatible_count}.'
    )

# MonoDETR also imports PyTorch's removed private _LinearWithBias class.
# Its only use is an attention output projection, for which public nn.Linear
# has the same parameters and checkpoint keys.
attention_source = ops_dir / 'modules/ms_deform_attn.py'
private_linear_import = (
    "if float(torch.__version__.split('.')[0]) == 0 or "
    "(float(torch.__version__.split('.')[0]) == 1 and "
    "float(torch.__version__.split('.')[1])) < 9:\n"
    "    from torch.nn.modules.linear import _LinearWithBias\n"
    "else:\n"
    "    from torch.nn.modules.linear import NonDynamicallyQuantizableLinear as _LinearWithBias"
)
public_linear_import = 'from torch.nn import Linear as _LinearWithBias'
attention_text = attention_source.read_text()
private_linear_count = attention_text.count(private_linear_import)
public_linear_count = attention_text.count(public_linear_import)
if private_linear_count == 1 and public_linear_count == 0:
    attention_source.write_text(attention_text.replace(private_linear_import, public_linear_import))
    print('Replaced removed private _LinearWithBias import with public torch.nn.Linear.')
elif private_linear_count == 0 and public_linear_count == 1:
    print('Public torch.nn.Linear compatibility patch is already applied.')
else:
    raise RuntimeError(
        'Unexpected MonoDETR attention source: expected one old or one patched '
        f'linear import, found old={private_linear_count}, patched={public_linear_count}.'
    )

# torch._overrides was private and no longer exists. MonoDETR's version check
# can select it on newer version strings, so use the public module directly.
private_overrides_import = (
    "if float(torch.__version__.split('.')[0]) == 0 or "
    "(float(torch.__version__.split('.')[0]) == 1 and "
    "float(torch.__version__.split('.')[1])) < 7:\n"
    "    from torch._overrides import has_torch_function, handle_torch_function\n"
    "else:\n"
    "    from torch.overrides import has_torch_function, handle_torch_function"
)
public_overrides_import = 'from torch.overrides import has_torch_function, handle_torch_function'
attention_text = attention_source.read_text()
private_overrides_count = attention_text.count(private_overrides_import)
public_overrides_count = attention_text.count(public_overrides_import)
if private_overrides_count == 1 and public_overrides_count == 1:
    attention_source.write_text(attention_text.replace(private_overrides_import, public_overrides_import))
    print('Replaced removed private torch._overrides fallback with public torch.overrides.')
elif private_overrides_count == 0 and public_overrides_count == 1:
    print('Public torch.overrides compatibility patch is already applied.')
else:
    raise RuntimeError(
        'Unexpected MonoDETR attention source: expected one compatibility block '
        f'or one public override import, found old_block={private_overrides_count}, '
        f'public_imports={public_overrides_count}.'
    )

# PyTorch 2.6 changed torch.load to weights_only=True by default. The pinned
# official checkpoint includes NumPy training metadata, so scope the legacy
# pickle behavior to MonoDETR's single checkpoint loader call.
checkpoint_loader_source = MONODETR_REPO / 'lib/helpers/save_helper.py'
implicit_checkpoint_load = 'checkpoint = torch.load(filename, map_location)'
explicit_checkpoint_load = 'checkpoint = torch.load(filename, map_location, weights_only=False)'
loader_text = checkpoint_loader_source.read_text()
implicit_load_count = loader_text.count(implicit_checkpoint_load)
explicit_load_count = loader_text.count(explicit_checkpoint_load)
if implicit_load_count == 1 and explicit_load_count == 0:
    checkpoint_loader_source.write_text(loader_text.replace(implicit_checkpoint_load, explicit_checkpoint_load))
    print('Made official MonoDETR checkpoint loading explicit for PyTorch 2.6+ compatibility.')
elif implicit_load_count == 0 and explicit_load_count == 1:
    print('PyTorch 2.6 checkpoint-loader compatibility patch is already applied.')
else:
    raise RuntimeError(
        'Unexpected MonoDETR checkpoint loader: expected one implicit or one explicit '
        f'torch.load call, found implicit={implicit_load_count}, explicit={explicit_load_count}.'
    )

# Remove only generated extension objects left by an earlier failed compile.
shutil.rmtree(ops_dir / 'build', ignore_errors=True)
run_streamed(
    [sys.executable, 'setup.py', 'build', 'install'],
    cwd=ops_dir,
    env={'MAX_JOBS': '2'},
)
run_streamed([
    sys.executable, '-c',
    "import torch, MultiScaleDeformableAttention; "
    "from lib.models.monodetr import build_monodetr; "
    "print('MonoDETR extension and full model import passed;', "
    "'torch=', torch.__version__, 'cuda=', torch.version.cuda, "
    "'device=', torch.cuda.get_device_name(0))",
], cwd=MONODETR_REPO)
print('MonoDETR commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=MONODETR_REPO, text=True).strip())

In [ ]:
# Download the official 20.83-moderate checkpoint once and prepare runtime config.
TEACHER_ROOT.mkdir(parents=True, exist_ok=True)
if not TEACHER_CHECKPOINT.is_file():
    run_streamed([
        sys.executable, '-m', 'gdown',
        '--id', OFFICIAL_CHECKPOINT_GDRIVE_ID,
        '--output', TEACHER_CHECKPOINT,
    ])
if TEACHER_CHECKPOINT.stat().st_size < 10_000_000:
    raise RuntimeError(f'Checkpoint looks incomplete: {TEACHER_CHECKPOINT.stat().st_size} bytes')

# This file comes from the pinned official Google Drive ID above. Because the
# legacy checkpoint contains pickled NumPy metadata, record its digest and
# validate its structure before the long inference run.
with TEACHER_CHECKPOINT.open('rb') as checkpoint_file:
    checkpoint_sha256 = hashlib.file_digest(checkpoint_file, 'sha256').hexdigest()
(TEACHER_ROOT / 'checkpoint_best.sha256').write_text(f'{checkpoint_sha256}  {TEACHER_CHECKPOINT.name}\n')
print('Official checkpoint SHA-256:', checkpoint_sha256)
import torch
checkpoint_probe = torch.load(TEACHER_CHECKPOINT, map_location='cpu', weights_only=False)
if not isinstance(checkpoint_probe, dict) or not isinstance(checkpoint_probe.get('model_state'), dict):
    raise RuntimeError('Official checkpoint did not contain the expected model_state dictionary.')
print('Checkpoint structure passed; model tensors:', len(checkpoint_probe['model_state']))
del checkpoint_probe

runtime_cfg = yaml.safe_load((MONODETR_REPO / 'configs/monodetr.yaml').read_text())
runtime_cfg['dataset']['root_dir'] = str(MONODETR_KITTI)
runtime_cfg['dataset']['train_split'] = 'train'
runtime_cfg['dataset']['test_split'] = 'val'
runtime_cfg['dataset']['batch_size'] = 4
runtime_cfg['dataset']['writelist'] = ['Car']
runtime_cfg['trainer']['gpu_ids'] = '0'
runtime_cfg['trainer']['save_path'] = 'outputs/'
runtime_cfg['trainer']['save_all'] = False
runtime_cfg['tester']['mode'] = 'single'
runtime_cfg['tester']['threshold'] = 0.001
runtime_cfg['tester']['topk'] = 150
runtime_config_path = MONODETR_REPO / 'configs/monodetr_mobileadas3d_teacher.yaml'
runtime_config_path.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False))

official_output_dir = MONODETR_REPO / 'outputs/monodetr'
official_output_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(TEACHER_CHECKPOINT, official_output_dir / 'checkpoint_best.pth')
print('Checkpoint:', TEACHER_CHECKPOINT)
print('Checkpoint size MB:', round(TEACHER_CHECKPOINT.stat().st_size / 1e6, 1))
print(runtime_config_path.read_text())

In [ ]:
# Run teacher inference. Expect 3,769 output files.
run_streamed([
    sys.executable, '-u', 'tools/train_val.py',
    '--config', 'configs/monodetr_mobileadas3d_teacher.yaml',
    '--evaluate_only',
], cwd=MONODETR_REPO)

prediction_dir = MONODETR_REPO / 'outputs/monodetr/outputs/data'
prediction_files = list(prediction_dir.glob('*.txt'))
print('Prediction directory:', prediction_dir)
print('Prediction files:', len(prediction_files))
if len(prediction_files) != 3769:
    raise RuntimeError('Teacher inference is incomplete; do not report partial AP.')

In [ ]:
# Evaluate with the same local AP_R40 implementation used for MobileADAS3D.
TEACHER_OUTPUT.mkdir(parents=True, exist_ok=True)
run_streamed([
    sys.executable, '-u', 'scripts/evaluate_kitti_prediction_dir.py',
    '--config', 'configs/kitti_mnv4_quality_scoring_v3.yaml',
    '--profile', 'colab_drive',
    '--dataset-root', MONODETR_KITTI,
    '--split-dir', SPLIT_DIR,
    '--prediction-dir', prediction_dir,
    '--split', 'val',
    '--classes', 'Car',
    '--source-name', f'MonoDETR_official_{MONODETR_COMMIT}',
    '--output-dir', TEACHER_OUTPUT,
], cwd=MOBILE_REPO)

summary_path = TEACHER_OUTPUT / 'kitti_r40_summary.json'
summary = json.loads(summary_path.read_text())
if not summary['complete_split']:
    raise RuntimeError('Teacher result is incomplete and cannot pass the feasibility gate.')
car_3d_moderate = next(
    row['ap_r40'] for row in summary['metrics']
    if row['metric'] == '3d' and row['class_name'] == 'Car' and row['difficulty'] == 'moderate'
)
print('\nTeacher Car 3D moderate AP_R40:', car_3d_moderate)
print('Feasibility gate (>=15):', 'PASS' if car_3d_moderate >= 15.0 else 'FAIL')
print('Summary:', summary_path)

## Teacher Task 2: reproducible Chen-train prediction cache

Run these cells only after the validation gate passes. This uses a separate MonoDETR output name, so it cannot overwrite the canonical validation predictions. The cache manifest is written with `complete: true` only after all 3,712 expected KITTI files have been parsed, copied to Drive, and checksum-verified.

In [ ]:
# Create a train-split runtime config and isolated local output directory.
TRAIN_CACHE_NAME = 'chen_train_20260731'
TRAIN_MODEL_NAME = 'monodetr_train_cache'
TRAIN_CACHE_OUTPUT = TEACHER_ROOT / TRAIN_CACHE_NAME
train_runtime_cfg = yaml.safe_load(runtime_config_path.read_text())
train_runtime_cfg['model_name'] = TRAIN_MODEL_NAME
train_runtime_cfg['dataset']['test_split'] = 'train'
train_runtime_config_path = MONODETR_REPO / 'configs/monodetr_mobileadas3d_teacher_train_cache.yaml'
train_runtime_config_path.write_text(yaml.safe_dump(train_runtime_cfg, sort_keys=False))
train_official_output_dir = MONODETR_REPO / 'outputs' / TRAIN_MODEL_NAME
train_official_output_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(TEACHER_CHECKPOINT, train_official_output_dir / 'checkpoint_best.pth')
print('Train cache output:', TRAIN_CACHE_OUTPUT)
print('Train runtime config:', train_runtime_config_path)
print(train_runtime_config_path.read_text())

In [ ]:
# Run MonoDETR on the 3,712-image Chen training split. This takes roughly
# the same time as validation. If Colab interrupts inference, rerun this cell.
run_streamed([
    sys.executable, '-u', 'tools/train_val.py',
    '--config', train_runtime_config_path.relative_to(MONODETR_REPO),
    '--evaluate_only',
], cwd=MONODETR_REPO)
train_prediction_dir = train_official_output_dir / 'outputs/data'
train_prediction_files = list(train_prediction_dir.glob('*.txt'))
print('Train prediction directory:', train_prediction_dir)
print('Train prediction files:', len(train_prediction_files))
if len(train_prediction_files) != 3712:
    raise RuntimeError('Train teacher inference is incomplete; cache was not created.')

In [ ]:
# Validate, copy to Drive, checksum, and write the completion manifest last.
run_streamed([
    sys.executable, '-u', 'scripts/create_teacher_prediction_cache.py',
    '--prediction-dir', train_prediction_dir,
    '--split-file', SPLIT_DIR / 'train.txt',
    '--output-dir', TRAIN_CACHE_OUTPUT,
    '--runtime-config', train_runtime_config_path,
    '--teacher-name', 'MonoDETR_official',
    '--teacher-source-commit', MONODETR_COMMIT,
    '--checkpoint-sha256', checkpoint_sha256,
    '--expected-count', '3712',
    '--allowed-classes', 'Car',
], cwd=MOBILE_REPO)
train_manifest_path = TRAIN_CACHE_OUTPUT / 'teacher_cache_manifest.json'
train_manifest = json.loads(train_manifest_path.read_text())
assert train_manifest['complete'] is True
assert train_manifest['split_images'] == 3712
assert train_manifest['prediction_files'] == 3712
assert train_manifest['checkpoint_sha256'] == checkpoint_sha256
print('Validated train teacher cache:', train_manifest_path)
print(json.dumps(train_manifest, indent=2))